# Prepare PRISM JSONL for the APC repository

This notebook converts `train.jsonl`, `dev.jsonl`, and `test.jsonl` from Kaggle Input into the repository's 4-line APC format:

```text
sentence with $T$ placeholder
aspect term
aspect category
sentiment
```

One explicit quad becomes one APC sample. Implicit quads with no aspect term are excluded from APC/ATE and counted in the report.

In [ ]:
from pathlib import Path
from collections import Counter
from datetime import datetime
import json
import re
import zipfile

# Change this to the Kaggle Input folder containing train.jsonl/dev.jsonl/test.jsonl.
RAW_INPUT_DIR = Path('/kaggle/input/prism-raw')

# Output can be used as DATA_INPUT_DIR in the multilingual training notebook.
OUTPUT_DIR = Path('/kaggle/working/prism_dataset_apc')
SPLITS = ('train', 'dev', 'test')

print('Raw input:', RAW_INPUT_DIR)
print('Output:', OUTPUT_DIR)

In [ ]:
def read_jsonl(path):
    records = []
    errors = []
    with path.open('r', encoding='utf-8') as handle:
        for line_no, line in enumerate(handle, 1):
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError as exc:
                errors.append((line_no, str(exc)))
                continue
            if not isinstance(row, dict):
                errors.append((line_no, 'record is not an object'))
                continue
            records.append(row)
    return records, errors


def replace_first_aspect(text, aspect):
    # Prefer exact matching so punctuation and casing remain unchanged.
    start = text.find(aspect)
    if start >= 0:
        return text[:start] + '$T$' + text[start + len(aspect):]

    # Fallback for casing differences between the annotation and sentence.
    match = re.search(re.escape(aspect), text, flags=re.IGNORECASE)
    if match:
        return text[:match.start()] + '$T$' + text[match.end():]
    return None


def normalize_sentiment(value):
    value = str(value or '').strip().lower()
    aliases = {'pos': 'positive', 'neg': 'negative', 'neu': 'neutral'}
    return aliases.get(value, value)


def convert_split(split):
    source = RAW_INPUT_DIR / f'{split}.jsonl'
    if not source.is_file():
        raise FileNotFoundError(f'Missing input file: {source}')

    records, errors = read_jsonl(source)
    samples = []
    stats = Counter()
    seen = set()

    for record in records:
        text = str(record.get('text') or '').strip()
        if not text:
            stats['empty_text'] += 1
            continue
        quads = record.get('quads') or []
        if not quads:
            stats['no_quads'] += 1
        for quad in quads:
            stats['quads_total'] += 1
            aspect = str(quad.get('aspect_term') or '').strip()
            if not aspect:
                stats['implicit_aspect_skipped'] += 1
                continue
            sentence = replace_first_aspect(text, aspect)
            if sentence is None:
                stats['aspect_not_found_skipped'] += 1
                continue
            category = str(quad.get('aspect_category') or 'UNKNOWN').strip().upper()
            sentiment = normalize_sentiment(quad.get('sentiment'))
            if sentiment not in {'positive', 'negative', 'neutral'}:
                stats['unknown_sentiment_skipped'] += 1
                continue
            key = (sentence, aspect, category, sentiment)
            if key in seen:
                stats['duplicate_skipped'] += 1
                continue
            seen.add(key)
            samples.append((sentence, aspect, category, sentiment))
            stats['samples_written'] += 1
            stats[f'category::{category}'] += 1
            stats[f'sentiment::{sentiment}'] += 1

    destination = OUTPUT_DIR / f'{split}.apc'
    destination.parent.mkdir(parents=True, exist_ok=True)
    with destination.open('w', encoding='utf-8', newline='\n') as handle:
        for sentence, aspect, category, sentiment in samples:
            handle.write(f'{sentence}\n{aspect}\n{category}\n{sentiment}\n')

    stats['records_read'] = len(records)
    stats['json_errors'] = len(errors)
    if errors:
        print(f'{split}: JSON errors:', errors[:3])
    return destination, stats

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
all_stats = {}
for split in SPLITS:
    path, stats = convert_split(split)
    all_stats[split] = stats
    print(f'\n{split}: {path}')
    for key, value in sorted(stats.items()):
        print(f'  {key}: {value}')

In [ ]:
# Validate the exact 4-line format consumed by common.dataset_utils.parse_apc_file.
def validate_apc(path):
    lines = path.read_text(encoding='utf-8').splitlines()
    if len(lines) % 4 != 0:
        raise ValueError(f'{path} has {len(lines)} lines, not a multiple of 4')
    samples = []
    for i in range(0, len(lines), 4):
        sentence, aspect, category, sentiment = lines[i:i + 4]
        if '$T$' not in sentence:
            raise ValueError(f'{path}:{i + 1} has no $T$ placeholder')
        if not aspect or not category or sentiment not in {'positive', 'negative', 'neutral'}:
            raise ValueError(f'Invalid sample at {path}:{i + 1}')
        samples.append((sentence, aspect, category, sentiment))
    return samples

for split in SPLITS:
    samples = validate_apc(OUTPUT_DIR / f'{split}.apc')
    print(f'{split}: {len(samples)} valid APC samples')

print('\nDataset ready for the training notebook:')
print(f'DATA_INPUT_DIR = {OUTPUT_DIR!s}')

In [ ]:
# Package the converted dataset as a Kaggle-downloadable artifact.
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
zip_path = Path('/kaggle/working') / f'prism_dataset_apc_{stamp}.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT_DIR.glob('*.apc')):
        archive.write(path, path.name)
print(f'Created: {zip_path}')
print(f'Size: {zip_path.stat().st_size / 1024:.1f} KB')